# Taller local: Data Engineer Junior Advanced
### Pipeline ETL con PySpark y Spark SQL en Entorno Local
**Rol:** Senior Data Engineer  
**Tecnologías:** PySpark 3.5+, Spark SQL, Parquet
**Caso de Negocio:** Ventas de una tienda retail latinoamericana.

## 0. Configuración del Entorno y Preprocesamiento
Configuramos las variables de entorno locales de Python y Hadoop (en Windows para soporte de winutils) y garantizamos que los datos de `taller_data_engineer.xlsx` estén extraídos en `datos/`.

In [ ]:
import os
import sys
import zipfile
import re
import csv
import json

# Variables de entorno para ejecución de PySpark en Windows
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
hadoop_home = os.path.abspath('hadoop')
if os.path.exists(os.path.join(hadoop_home, 'bin', 'winutils.exe')):
    os.environ['HADOOP_HOME'] = hadoop_home
    os.environ['hadoop.home.dir'] = hadoop_home
    os.environ['PATH'] = os.path.join(hadoop_home, 'bin') + ';' + os.environ.get('PATH', '')

print("Entorno local configurado exitosamente.")

## 1. Inicialización de SparkSession y Lectura de Datos
- `master('local[*]')` con `appName('taller-data-engineer')`.
- Lectura de CSVs desactivando la inferencia automática (`inferSchema=False`) y con `header=True`.
- Conversión explícita de tipos de datos antes de operar.

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import (
    col, when, to_date, to_timestamp, round as spark_round,
    sum as spark_sum, count, countDistinct, date_format,
    row_number, lit, broadcast
)
from pyspark.sql.types import (
    StringType, IntegerType, DoubleType, BooleanType
)

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("taller-data-engineer")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession inicializada: v{spark.version}")

# Lectura sin inferSchema
raw_customers = spark.read.option("header", True).option("inferSchema", False).csv("datos/customers.csv")
raw_products = spark.read.option("header", True).option("inferSchema", False).csv("datos/products.csv")
raw_orders = spark.read.option("header", True).option("inferSchema", False).csv("datos/orders.csv")
raw_order_items = spark.read.option("header", True).option("inferSchema", False).csv("datos/order_items.csv")

# Conversión explícita de tipos
customers = raw_customers.select(
    col("customer_id").cast(StringType()),
    col("first_name").cast(StringType()),
    col("last_name").cast(StringType()),
    col("email").cast(StringType()),
    col("country").cast(StringType()),
    to_date(col("signup_date"), "yyyy-MM-dd").alias("signup_date"),
    col("segment").cast(StringType())
)

products = raw_products.select(
    col("product_id").cast(StringType()),
    col("product_name").cast(StringType()),
    col("category").cast(StringType()),
    col("unit_price").cast(DoubleType()),
    col("active").cast(BooleanType())
)

orders = raw_orders.select(
    col("order_id").cast(StringType()),
    col("customer_id").cast(StringType()),
    to_timestamp(col("order_ts")).alias("order_ts"),
    col("status").cast(StringType()),
    col("channel").cast(StringType()),
    to_timestamp(col("updated_at")).alias("updated_at")
)

order_items = raw_order_items.select(
    col("line_id").cast(StringType()),
    col("order_id").cast(StringType()),
    col("product_id").cast(StringType()),
    col("quantity").cast(IntegerType()),
    col("discount_pct").cast(DoubleType())
)

orders.printSchema()

## 2. Perfilamiento y Diagnóstico de Calidad
Tratamiento justificado de anomalías encontradas:
- **Duplicados:** Deduplicación canónica (`customer_id`, `order_id`). -> **CORREGIR**
- **Typo en dominio:** `status = 'deliverd'` a `'delivered'`. -> **CORREGIR**
- **Fechas mal formadas:** `order_ts` no parseable (`2026-99-31`). -> **CUARENTENA**
- **Integridad referencial:** IDs inexistentes o nulos en clientes, pedidos y productos. -> **CUARENTENA**
- **Reglas de negocio (Dominio numérico):** `quantity <= 0` o `discount_pct > 1.0`. -> **RECHAZAR**

In [ ]:
# 1. Deduplicación canónica
w_cust = Window.partitionBy("customer_id").orderBy(col("signup_date").desc())
clean_customers = customers.withColumn("rn", row_number().over(w_cust)).filter(col("rn") == 1).drop("rn")

w_ord = Window.partitionBy("order_id").orderBy(col("updated_at").desc())
dedup_orders = orders.withColumn("rn", row_number().over(w_ord)).filter(col("rn") == 1).drop("rn")

# 2. Corrección del error tipográfico
corrected_orders = dedup_orders.withColumn(
    "status", when(col("status") == "deliverd", "delivered").otherwise(col("status"))
)

# 3. Cuarentena de pedidos con fecha inválida
quarantine_ts_orders = corrected_orders.filter(col("order_ts").isNull()).withColumn("quarantine_reason", lit("MALFORMED_ORDER_TIMESTAMP"))
valid_ts_orders = corrected_orders.filter(col("order_ts").isNotNull())

# 4. Integridad referencial: pedidos -> clientes
orders_cust_joined = valid_ts_orders.join(
    broadcast(clean_customers.select("customer_id").withColumn("cust_exists", lit(True))),
    on="customer_id", how="left"
)
quarantine_orphan_orders = (
    orders_cust_joined
    .filter(col("cust_exists").isNull())
    .withColumn("quarantine_reason", when(col("customer_id").isNull() | (col("customer_id") == ""), "NULL_CUSTOMER").otherwise("ORPHAN_CUSTOMER"))
    .drop("cust_exists")
)
clean_orders = orders_cust_joined.filter(col("cust_exists").isNotNull()).drop("cust_exists")

# 5. Productos con precios válidos
quarantine_products = products.filter(col("unit_price").isNull() | (col("unit_price") <= 0)).withColumn(
    "quarantine_reason", when(col("unit_price").isNull(), "NULL_PRICE").otherwise("NEGATIVE_PRICE")
)
clean_products = products.filter(col("unit_price").isNotNull() & (col("unit_price") > 0))

# 6. Rechazo de ítems con cantidades <= 0 o descuentos > 1.0
rejected_items = order_items.filter(
    col("quantity").isNull() | (col("quantity") <= 0) |
    col("discount_pct").isNull() | (col("discount_pct") < 0.0) | (col("discount_pct") > 1.0)
).withColumn(
    "rejection_reason", when(col("quantity").isNull() | (col("quantity") <= 0), "INVALID_QUANTITY").otherwise("INVALID_DISCOUNT_PCT")
)
valid_domain_items = order_items.filter(
    col("quantity").isNotNull() & (col("quantity") > 0) &
    col("discount_pct").isNotNull() & (col("discount_pct") >= 0.0) & (col("discount_pct") <= 1.0)
)

# 7. Integridad referencial de ítems con pedidos y productos
items_fk = valid_domain_items.join(
    broadcast(clean_orders.select("order_id").withColumn("ord_exists", lit(True))), on="order_id", how="left"
).join(
    broadcast(clean_products.select("product_id").withColumn("prod_exists", lit(True))), on="product_id", how="left"
)
quarantine_items = items_fk.filter(col("ord_exists").isNull() | col("prod_exists").isNull()).withColumn(
    "quarantine_reason", when(col("ord_exists").isNull() & col("prod_exists").isNull(), "ORPHAN_ORDER_AND_PRODUCT")
    .when(col("ord_exists").isNull(), "ORPHAN_ORDER_OR_PARENT_IN_QUARANTINE").otherwise("ORPHAN_PRODUCT")
).drop("ord_exists", "prod_exists")
clean_order_items = items_fk.filter(col("ord_exists").isNotNull() & col("prod_exists").isNotNull()).drop("ord_exists", "prod_exists")

print(f"Diagnóstico final: Limpios={clean_order_items.count()}, Cuarentena={quarantine_items.count()}, Rechazados={rejected_items.count()}")

## 3. Limpieza, Joins y Cálculo del Importe Neto
Fórmula requerida:
$$\text{net\_amount} = \text{quantity} \times \text{unit\_price} \times (1 - \text{discount\_pct})$$

In [ ]:
sales_detail = (
    clean_order_items
    .join(clean_orders.select("order_id", "customer_id", "order_ts", "status", "channel"), on="order_id", how="inner")
    .join(broadcast(clean_customers.select("customer_id", "country", "segment")), on="customer_id", how="inner")
    .join(broadcast(clean_products.select("product_id", "product_name", "category", "unit_price")), on="product_id", how="inner")
    .withColumn(
        "net_amount",
        spark_round(col("quantity") * col("unit_price") * (lit(1.0) - col("discount_pct")), 2)
    )
)

sales_detail.select("line_id", "order_id", "country", "channel", "category", "quantity", "unit_price", "discount_pct", "net_amount").show(5, truncate=False)

## 4. Agregaciones de Negocio
Ventas netas por **mes, país, canal y categoría**.

In [ ]:
monthly_sales = (
    sales_detail
    .withColumn("order_month", date_format(col("order_ts"), "yyyy-MM"))
    .groupBy("order_month", "country", "channel", "category")
    .agg(
        spark_round(spark_sum("net_amount"), 2).alias("total_net_amount"),
        spark_sum("quantity").alias("total_units"),
        countDistinct("order_id").alias("unique_orders"),
        count("line_id").alias("total_lines")
    )
    .orderBy("order_month", "country", "channel", "category")
)

monthly_sales.show(10, truncate=False)

## 5. Reto Incremental Idempotente
- Lectura de lotes JSON Lines (`orders_incremental.jsonl`, `order_items_incremental.jsonl`).
- Deduplicación temporal por `order_id` con precedencia de `updated_at DESC`.
- Garantía de idempotencia: reprocesar el mismo lote genera exactamente el mismo estado.

In [ ]:
raw_inc_orders = spark.read.json("datos/incremental/orders_incremental.jsonl")
raw_inc_items = spark.read.json("datos/incremental/order_items_incremental.jsonl")

inc_orders = raw_inc_orders.select(
    col("order_id").cast(StringType()),
    col("customer_id").cast(StringType()),
    to_timestamp(col("order_ts")).alias("order_ts"),
    when(col("status") == "deliverd", "delivered").otherwise(col("status")).alias("status"),
    col("channel").cast(StringType()),
    to_timestamp(col("updated_at")).alias("updated_at")
)

inc_items = raw_inc_items.select(
    col("line_id").cast(StringType()),
    col("order_id").cast(StringType()),
    col("product_id").cast(StringType()),
    col("quantity").cast(IntegerType()),
    col("discount_pct").cast(DoubleType())
)

# Merge histórico + incremental
w_dedup_order = Window.partitionBy("order_id").orderBy(col("updated_at").desc())
idempotent_orders = (
    orders.unionByName(inc_orders)
    .withColumn("rn", row_number().over(w_dedup_order))
    .filter(col("rn") == 1)
    .drop("rn")
)

print(f"Pedidos consolidados: {idempotent_orders.count()} (20.000 + 100)")
# Comprobar que el pedido O00005 tiene la versión tardía más reciente
idempotent_orders.filter(col("order_id") == "O00005").show(truncate=False)

## 6. Persistencia y Controles de Calidad (QA)
- Guardado en formato Parquet en `output/`.
- Controles de Calidad:
  1. Reconciliación matemática entre detalle y agregado ($95,297,643.79).
  2. Unicidad de clave primaria `line_id`.
  3. Balance contable de conservación de filas (Raw = Clean + Rej + Quar).

In [ ]:
import os
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

sales_detail.write.mode("overwrite").parquet(os.path.join(output_dir, "sales_detail.parquet"))
monthly_sales.write.mode("overwrite").parquet(os.path.join(output_dir, "monthly_sales.parquet"))
print("Resultados persistidos exitosamente en Parquet.")

# QA 1: Reconciliación
sum_detail = sales_detail.agg(spark_sum("net_amount")).collect()[0][0]
sum_agg = monthly_sales.agg(spark_sum("total_net_amount")).collect()[0][0]
diff = abs(sum_detail - sum_agg)
print(f"[QA 1] Reconciliación: Detalle=${sum_detail:,.2f} | Agregado=${sum_agg:,.2f} | Diff={diff:.4f}")
assert diff < 0.01, "Fallo en QA 1"

# QA 2: Unicidad
n_rows = sales_detail.count()
n_distinct_lines = sales_detail.select(countDistinct("line_id")).collect()[0][0]
print(f"[QA 2] Unicidad: Filas={n_rows} | Unique lines={n_distinct_lines}")
assert n_rows == n_distinct_lines, "Fallo en QA 2"

# QA 3: Balance de filas
raw_total = raw_order_items.count()
clean_total = clean_order_items.count()
rej_total = rejected_items.count()
quar_total = quarantine_items.count()
print(f"[QA 3] Balance: Raw ({raw_total}) == Clean ({clean_total}) + Rej ({rej_total}) + Quar ({quar_total})")
assert raw_total == clean_total + rej_total + quar_total, "Fallo en QA 3"

print("¡TODOS LOS CONTROLES DE CALIDAD SUPERADOS EXITOSAMENTE!")